This script set a 2D recentangle lattice with unit cell basis $\bm{a}$ [9.19310, 0] angstrom, and [0, 5.2586] angstrom, the sublattice positions are (0, 0) and (0.5, 0.5) in the unit of $\bm{a}$.

It builds a spin Hamiltonian with exchange interactions and Zeeman term. The superlattice is $1 \times 1$.

In [ ]:
import numpy as np
import spintoolkit_py as sptk

In [ ]:
# define units
unit0 = sptk.units(energy="K", length="angstrom")

In [ ]:
# the length unit is angstrom
# 2D lattice vectors
a = np.array([
    [9.19310 * unit0.angstrom, 0.0],
    [0.0, 5.25860 * unit0.angstrom]
])

# superlattice A = 1 x 1
A = np.array([1, 1])

# the sublattice positions
pos_sub = np.array([
                    [0.0, 0.0],
                    [0.5, 0.5]])

In [ ]:
# define the lattice
latt = sptk.lattice(basis_a=a, pos_sub=pos_sub, l = A)

# spin-5/2
hamiltonian = sptk.model_spin(S=2.5, mode="dipole", lattice=latt)

L       = 2
num_sub = 2
dim     = 2
Real space basis (a):                              
a0: [              9.1931                   0]
a1: [                   0              5.2586]
Reciprocal space basis (b / 2π):                   
b0: [           0.1087772                   0]
b1: [                   0           0.1901647]
Superlattice real space basis (A) [unit: a]:       
A0: [                   1                   0]
A1: [                   0                   1]
Superlattice reciprocal space basis (B) [unit: b]: 
B0: [                   1                   0]
B1: [                   0                   1]
Sublattice positions [unit: a]: 
sub[    0] = [              0              0]
sub[    1] = [            0.5            0.5]
Minimal bond length: 5.2586

Model with S=5/2 (mode = dipole) initialized.


In [ ]:
# pre-compute r_shift to consider periodic images
r_shift = []
for j in range(-1, 2):
    for i in range(-1, 2):
        r_shift.append([i * A[0], j * A[1]])

In [7]:
J = 1.0 * unit0.K  # exchange interaction in Kelvin units
B = 1.0 * unit0.T  # magnetic field in Tesla units
g = 2.0  # g-factor
total_sites = latt.total_sites()

In [ ]:
# -----------------------------------------
# loop over all sites i
# -----------------------------------------
for site_i in range(total_sites):
    coor_i, sub_i = latt.site2coor(site=site_i)	# find the coordiante use lattice basis
    coor0_i, r̃i   = latt.r2superlattice(coor=coor_i)
    cart_i        = latt.coor2cart(coor=coor_i, sub=sub_i)	# find the cartesian coordinate

	# the number of J bonds
    cnt1 = 0

    # -----------------------------------------
    # loop: neighbors j (+ periodic images)
    # -----------------------------------------
    for site_j in range(total_sites):
        coor0_j, sub0_j = latt.site2coor(site=site_j)

        for r_shift0 in r_shift:
            # apply shift
            coor_j      = [coor0_j[0] + r_shift0[0], coor0_j[1] + r_shift0[1]]
            coor0_j, r̃j = latt.r2superlattice(coor=coor_j)
            cart_j      = latt.coor2cart(coor=coor_j, sub=sub0_j)

            # distance
            dx = cart_i[0] - cart_j[0]
            dy = cart_i[1] - cart_j[1]
            r = np.sqrt(dx*dx + dy*dy)

            # -----------------------------------------
            # J bond (r = 5.2586)
            # 0.5 * J to avoid double counting
            # np.abs(r - 5.2586) < 0.01
            # -----------------------------------------
            if np.abs(r - 5.2586) < 0.01:
                hamiltonian.add_2spin_XYZ(
                    J=sptk.Vec3(0.5 * J, 0.5 * J, 0.5 * J),
                    site_i=site_i, site_j=site_j,
                    rtilde_i=r̃i, rtilde_j=r̃j
                )
                cnt1 += 1

    # the number of J bonds for each site is 2
    assert cnt1 == 2

# -----------------------------------------
# Add Zeeman term
# -----------------------------------------
for site_i in range(total_sites):
    hamiltonian.add_zeeman(
        h=sptk.Vec3(0.0, 0.0, B * g),
        site=site_i
    )